In [ ]:
#### %cd /kaggle/working
!rm -rf anuraset
!git clone https://github.com/soundclim/anuraset.git
%cd anuraset
!pip install -q pyyaml pandas numpy scikit-learn librosa tqdm matplotlib torchmetrics
print("repo ready:", __import__("os").path.exists("baseline/configs/exp_resnet18.yaml"))

In [ ]:
import os, glob, pandas as pd
base  = "/kaggle/input/datasets/mismaresenka/anuraset-preprocessed/anuraset"
AUDIO = os.path.join(base, "audio")
stem2rel = {os.path.splitext(os.path.basename(p))[0]: os.path.relpath(p, AUDIO)
            for p in glob.glob(os.path.join(AUDIO, "**", "*.wav"), recursive=True)}
df = pd.read_csv(os.path.join(base, "metadata.csv"))
def key(r): return f"{r.fname}_{int(float(r.min_t))}_{int(float(r.max_t))}"
frac = df.apply(key, axis=1).isin(stem2rel).mean()
print(f"match: {frac:.1%}")
assert frac > 0.95, "STOP — filename match failed, tell Claude"
df[df.columns[0]] = df.apply(lambda r: stem2rel.get(key(r)), axis=1)
assert df[df.columns[0]].notna().all(), "STOP — some rows unmatched"
root = "/kaggle/working/anuraset_data"; os.makedirs(root, exist_ok=True)
df.to_csv(os.path.join(root, "metadata.csv"), index=False)
if not os.path.exists(os.path.join(root, "audio")): os.symlink(AUDIO, os.path.join(root, "audio"))
print("✓ data fixed —", len(df), "rows resolve")

In [ ]:
import yaml, shutil, os
# wipe the leftover smoke-test model so training starts clean
shutil.rmtree("/kaggle/working/anuraset/baseline/model_states", ignore_errors=True)
cfgp = "/kaggle/working/anuraset/baseline/configs/exp_resnet18.yaml"
cfg = yaml.safe_load(open(cfgp))
cfg.update({"data_root": "/kaggle/working/anuraset_data", "num_epochs": 10, "batch_size": 16, "model_type": "resnet50"})
yaml.safe_dump(cfg, open(cfgp, "w"))
print("config set for REAL run: 10 epochs")

In [ ]:
import subprocess, sys, os
os.chdir("/kaggle/working/anuraset")
print("training for real — this takes a while...\n" + "="*45)
r = subprocess.run([sys.executable, "baseline/train.py", "--config",
                    "baseline/configs/exp_resnet18.yaml"], capture_output=True, text=True)
print(r.stdout[-3000:]); print("STDERR:", r.stderr[-1500:]); print("="*45, "EXIT:", r.returncode)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# AnuraSet v2 — FIXED. Paste as ONE cell, replacing the previous big cell.
# Change vs last version: soundfile+scipy instead of librosa (resampy is
# missing on this image), and load failures now RAISE instead of returning
# silence. Plus a guard that stops before training if embeddings are flat.
# ═══════════════════════════════════════════════════════════════════

ROOT   = "/kaggle/input/datasets/mismaresenka/anuraset-preprocessed/anuraset"
AUDIO  = f"{ROOT}/audio"
META   = f"{ROOT}/metadata.csv"
CACHE  = "/kaggle/working/embeddings.npz"

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import tensorflow as tf
for _g in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(_g, True)

import gc, math, numpy as np, pandas as pd, soundfile as sf, torch
import torch.nn as nn, torch.nn.functional as F
from scipy.signal import resample_poly
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import f1_score, average_precision_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SR, WIN = 32000, 160000
torch.manual_seed(42); np.random.seed(42)

# ─── data ──────────────────────────────────────────────────────────
df = pd.read_csv(META)
LABELS = df.columns[df.columns.get_loc("subset") + 1:].tolist()
assert len(LABELS) == 42, f"expected 42 species, got {len(LABELS)}"

df["_path"] = (AUDIO + "/" + df["site"].astype(str) + "/" + df["fname"].astype(str)
               + "_" + df["min_t"].astype(str) + "_" + df["max_t"].astype(str) + ".wav")
assert os.path.exists(df["_path"].iloc[0])
Y = df[LABELS].values.astype(np.float32)
print(f"{len(df)} clips | {df.site.nunique()} sites | {dict(df.subset.value_counts())}")

# ─── loader: soundfile + polyphase resample, NO silent fallback ────
def load_one(p):
    w, sr = sf.read(p, dtype="float32", always_2d=False)
    if w.ndim > 1:
        w = w.mean(1)
    if sr != SR:
        g = math.gcd(SR, sr)
        w = resample_poly(w, SR // g, sr // g).astype(np.float32)
    out = np.zeros(WIN, np.float32)
    out[:min(len(w), WIN)] = w[:WIN]
    return out

# fail fast, loudly, before spending 8 minutes
_probe = np.stack([load_one(p) for p in df["_path"].iloc[[0, 1000, 50000]]])
assert _probe.std() > 1e-4, "audio still loading as silence — stop and report"
print(f"loader ok | std {_probe.std():.5f} | native sr {sf.info(df['_path'].iloc[0]).samplerate}")

# ─── stage 1: embeddings ───────────────────────────────────────────
def embed_all():
    import tensorflow_hub as hub, kagglehub, time
    N, B = len(df), 64
    emb = np.zeros((N, 1280), np.float32)
    model = hub.load(kagglehub.model_download(
        "google/bird-vocalization-classifier/tensorFlow2/bird-vocalization-classifier"))
    paths, pool, t0 = df["_path"].values, ThreadPoolExecutor(8), time.time()

    for s in range(0, N, B):
        batch = np.stack(list(pool.map(load_one, paths[s:s + B])))
        r = model.infer_tf(batch)
        emb[s:s + len(batch)] = np.asarray(
            r["embedding"] if isinstance(r, dict) else r[1])
        if s == 0:
            assert emb[:len(batch)].std(0).mean() > 1e-3, \
                "embeddings flat on first batch — wrong output tensor"
        if (s // B) % 50 == 0 and s:
            rate = (s + len(batch)) / (time.time() - t0)
            print(f"  {s+len(batch)}/{N} ({rate:.0f}/s, ~{(N-s)/rate/60:.0f} min left)")

    np.savez_compressed(CACHE, emb=emb)
    del model; tf.keras.backend.clear_session(); gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

embed_all()
X = np.load(CACHE)["emb"]

# ─── guard: this is what should have caught the last run ───────────
n_uniq = len(np.unique(X[:2000].round(3), axis=0))
print(f"embeddings {X.shape} | per-dim std {X.std(0).mean():.4f} | uniq/2000 {n_uniq}")
assert n_uniq > 1500, "embeddings not varied — do not trust anything below"

tr = (df.subset == "train").values
te = (df.subset == "test").values

# ─── stage 2: head ─────────────────────────────────────────────────
class FocalBCE(nn.Module):
    def __init__(self, pw, gamma=2.0):
        super().__init__(); self.gamma = gamma; self.register_buffer("pw", pw)
    def forward(self, logits, t):
        bce = F.binary_cross_entropy_with_logits(logits, t, reduction="none",
                                                 pos_weight=self.pw)
        p = torch.sigmoid(logits); pt = p * t + (1 - p) * (1 - t)
        return (bce * (1 - pt).pow(self.gamma)).mean()

def train(Xtr, Ytr, Xva, Yva, epochs=40, lr=1e-3, gamma=2.0, hidden=512,
          dropout=0.3, mixup=True, pw_cap=20):
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6          # standardize features
    Xtr, Xva = (Xtr - mu) / sd, (Xva - mu) / sd
    pos = Ytr.sum(0)
    pw = torch.tensor(np.clip((len(Ytr) - pos) / np.maximum(pos, 1), 1, pw_cap),
                      dtype=torch.float32, device=DEVICE)
    crit = FocalBCE(pw, gamma).to(DEVICE)
    m = nn.Sequential(nn.LayerNorm(Xtr.shape[1]), nn.Linear(Xtr.shape[1], hidden),
                      nn.GELU(), nn.Dropout(dropout),
                      nn.Linear(hidden, Ytr.shape[1])).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=1e-2)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    Xt = torch.tensor(Xtr, device=DEVICE); Yt = torch.tensor(Ytr, device=DEVICE)
    Xv = torch.tensor(Xva, device=DEVICE)
    dl = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(Xt, Yt),
                                     batch_size=512, shuffle=True)
    keep = Yva.sum(0) > 0                     # ignore zero-support classes
    best_ap, best = -1, None
    for ep in range(epochs):
        m.train()
        for xb, yb in dl:
            if mixup:
                lam = np.random.beta(0.4, 0.4)
                pm = torch.randperm(xb.size(0), device=DEVICE)
                xb = lam * xb + (1 - lam) * xb[pm]
                yb = torch.clamp(yb + yb[pm], 0, 1)
            opt.zero_grad(); crit(m(xb), yb).backward(); opt.step()
        sch.step(); m.eval()
        with torch.no_grad():
            pr = torch.sigmoid(m(Xv)).cpu().numpy()
        ap = average_precision_score(Yva[:, keep], pr[:, keep], average="macro")
        if ep % 10 == 0: print(f"  ep{ep:>3} macro-AP {ap:.4f}")
        if ap > best_ap: best_ap, best = ap, pr
    return best_ap, best

def tune(Yva, pr, grid=np.arange(0.02, 0.98, 0.01)):
    th = np.full(Yva.shape[1], 0.5)
    for c in range(Yva.shape[1]):
        if Yva[:, c].sum() == 0: continue
        th[c] = grid[int(np.argmax([f1_score(Yva[:, c], (pr[:, c] >= t).astype(int),
                                             zero_division=0) for t in grid]))]
    return th

ap, probs = train(X[tr], Y[tr], X[te], Y[te])
th   = tune(Y[te], probs)
keep = Y[te].sum(0) > 0
f05 = f1_score(Y[te][:, keep], (probs >= 0.5).astype(int)[:, keep],
               average="macro", zero_division=0)
fth = f1_score(Y[te][:, keep], (probs >= th).astype(int)[:, keep],
               average="macro", zero_division=0)

print(f"\nchance macro-AP     : {Y[te][:, keep].mean():.4f}   <- must beat this")
print(f"published baseline  : 0.2290")
print(f"macro-AP            : {ap:.4f}")
print(f"macro-F1 @ 0.5      : {f05:.4f}")
print(f"macro-F1 @ per-class: {fth:.4f}   (+{fth - f05:.4f})")
print(f"(over {keep.sum()} species with test support; {(~keep).sum()} excluded)")

per_class = pd.DataFrame({
    "species": LABELS, "support": Y[te].sum(0).astype(int), "threshold": th,
    "f1": f1_score(Y[te], (probs >= th).astype(int), average=None, zero_division=0),
}).sort_values("support", ascending=False)
display(per_class)


In [ ]:
import numpy as np
from sklearn.metrics import f1_score

# ---- 1. LEAKAGE: does any parent recording span both splits?
tr_f = set(df.loc[df.subset == "train", "fname"])
te_f = set(df.loc[df.subset == "test",  "fname"])
print(f"train recordings {len(tr_f)} | test {len(te_f)} | SHARED {len(tr_f & te_f)}")

# ---- 2. honest protocol: hold out recordings from train for tuning
def train_model(Xtr, Ytr, Xva, Yva, epochs=40, gamma=2.0, hidden=512):
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6
    Xtr_, Xva_ = (Xtr - mu) / sd, (Xva - mu) / sd
    pos = Ytr.sum(0)
    pw = torch.tensor(np.clip((len(Ytr) - pos) / np.maximum(pos, 1), 1, 20),
                      dtype=torch.float32, device=DEVICE)
    crit = FocalBCE(pw, gamma).to(DEVICE)
    m = nn.Sequential(nn.LayerNorm(Xtr.shape[1]), nn.Linear(Xtr.shape[1], hidden),
                      nn.GELU(), nn.Dropout(0.3),
                      nn.Linear(hidden, Ytr.shape[1])).to(DEVICE)
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-2)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    Xt = torch.tensor(Xtr_, device=DEVICE); Yt = torch.tensor(Ytr, device=DEVICE)
    dl = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(Xt, Yt),
                                     batch_size=512, shuffle=True)
    for _ in range(epochs):
        m.train()
        for xb, yb in dl:
            lam = np.random.beta(0.4, 0.4)
            pm = torch.randperm(xb.size(0), device=DEVICE)
            xb = lam * xb + (1 - lam) * xb[pm]
            yb = torch.clamp(yb + yb[pm], 0, 1)
            opt.zero_grad(); crit(m(xb), yb).backward(); opt.step()
        sch.step()
    m.eval()
    def predict(Z):
        with torch.no_grad():
            return torch.sigmoid(m(torch.tensor((Z - mu) / sd, dtype=torch.float32,
                                                device=DEVICE))).cpu().numpy()
    return predict

rng = np.random.default_rng(0)
recs = np.array(sorted(tr_f))
val_recs = set(rng.choice(recs, int(0.15 * len(recs)), replace=False))
isval = (df.subset == "train").values &  df.fname.isin(val_recs).values
istr  = (df.subset == "train").values & ~df.fname.isin(val_recs).values
te    = (df.subset == "test").values
print(f"subtrain {istr.sum()} | tune-val {isval.sum()} | test {te.sum()}")

pred = train_model(X[istr], Y[istr], X[isval], Y[isval])
th_honest = tune(Y[isval], pred(X[isval]))       # thresholds from VAL only
p_te = pred(X[te])

keep = (Y[te].sum(0) > 0) & (Y[isval].sum(0) > 0)
f_honest = f1_score(Y[te][:, keep], (p_te >= th_honest).astype(int)[:, keep],
                    average="macro", zero_division=0)
f_cheat  = f1_score(Y[te][:, keep], (p_te >= tune(Y[te], p_te)).astype(int)[:, keep],
                    average="macro", zero_division=0)
print(f"\nmacro-F1 (val-tuned, HONEST) : {f_honest:.4f}   <- the reportable number")
print(f"macro-F1 (test-tuned, inflated): {f_cheat:.4f}")

In [ ]:
keep38 = Y[te].sum(0) > 0
print("species in honest score:", keep.sum(), "| with test support:", keep38.sum())

scores = []
for seed in [0, 1, 2]:
    torch.manual_seed(seed); np.random.seed(seed)
    r = np.random.default_rng(seed)
    vr = set(r.choice(recs, int(0.15 * len(recs)), replace=False))
    iv = (df.subset == "train").values &  df.fname.isin(vr).values
    it = (df.subset == "train").values & ~df.fname.isin(vr).values
    pr = train_model(X[it], Y[it], X[iv], Y[iv])
    th = tune(Y[iv], pr(X[iv]))
    s = f1_score(Y[te][:, keep38], (pr(X[te]) >= th).astype(int)[:, keep38],
                 average="macro", zero_division=0)
    print(f"seed {seed}: macro-F1 over {keep38.sum()} species = {s:.4f}")
    scores.append(s)
print(f"\nREPORT: {np.mean(scores):.3f} ± {np.std(scores):.3f}")

In [ ]:
import pandas as pd
m = pd.read_csv("/kaggle/input/datasets/mismaresenka/anuraset-preprocessed/anuraset/metadata.csv")
print(m.shape)
print(m.columns.tolist())
m.head(3)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import f1_score

# --- plug in your objects ---
# y_true: (n_test, 42) binary array
# preds_by_seed: list of 3 (n_test, 42) binary arrays, val-tuned thresholds applied
# species: list of 42 species codes, column order matching y_true
# train_df: train split with one binary column per species

support = y_true.sum(axis=0)
keep = support > 0                      # the 38 test-supported species

f1s = np.stack([f1_score(y_true, p, average=None, zero_division=0) for p in preds_by_seed])
df = pd.DataFrame({
    "species": np.array(species)[keep],
    "f1_mean": f1s.mean(0)[keep],
    "f1_std":  f1s.std(0)[keep],
    "train_n": train_df[species].sum().values[keep],
    "test_n":  support[keep],
}).sort_values("f1_mean", ascending=False)

assert np.isclose(df.f1_mean.mean(), 0.628, atol=0.005)  # sanity check vs reported number

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(df.species, df.f1_mean, yerr=df.f1_std, capsize=2, color="#3a7d44")
ax.axhline(df.f1_mean.mean(), ls="--", c="k", lw=1, label=f"macro-F1 = {df.f1_mean.mean():.3f}")
for i, n in enumerate(df.train_n):
    ax.text(i, 0.02, f"{n:,}", rotation=90, ha="center", va="bottom", fontsize=7, color="white")
ax.set_ylabel("F1 (mean ± std, 3 seeds)")
ax.set_title("Per-species F1 on AnuraSet test set (Perch embeddings); bar labels = train clips")
ax.tick_params(axis="x", rotation=90, labelsize=8)
ax.set_ylim(0, 1); ax.legend()
plt.tight_layout(); plt.savefig("per_species_f1.png", dpi=200)

print(df[["f1_mean", "train_n"]].corr(method="spearman"))

In [ ]:
keep38 = Y[te].sum(0) > 0
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

per_seed_f1 = []
for seed in [0, 1, 2]:
    torch.manual_seed(seed); np.random.seed(seed)
    r = np.random.default_rng(seed)
    vr = set(r.choice(recs, int(0.15 * len(recs)), replace=False))
    iv = (df.subset == "train").values &  df.fname.isin(vr).values
    it = (df.subset == "train").values & ~df.fname.isin(vr).values
    pr = train_model(X[it], Y[it], X[iv], Y[iv])
    th = tune(Y[iv], pr(X[iv]))
    per_seed_f1.append(f1_score(Y[te], (pr(X[te]) >= th).astype(int),
                                average=None, zero_division=0))
    print(f"seed {seed} done")

f1s = np.stack(per_seed_f1)
res = pd.DataFrame({
    "species": LABELS,
    "f1_mean": f1s.mean(0), "f1_std": f1s.std(0),
    "train_n": Y[(df.subset == "train").values].sum(0).astype(int),
    "test_n":  Y[te].sum(0).astype(int),
})[keep38].sort_values("f1_mean", ascending=False)
print("macro-F1:", round(res.f1_mean.mean(), 3), "(should be ~0.628)")

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(res.train_n, res.f1_mean, color="#3a7d44")
ax.errorbar(res.train_n, res.f1_mean, yerr=res.f1_std, fmt="none", ecolor="gray", alpha=0.5)
for _, row in res.iterrows():
    ax.annotate(row.species, (row.train_n, row.f1_mean), fontsize=6, xytext=(3, 2), textcoords="offset points")
ax.set_xscale("log"); ax.set_ylim(0, 1)
ax.set_xlabel("Training clips (log scale)"); ax.set_ylabel("Test F1 (mean ± std, 3 seeds)")
ax.set_title(f"Per-species F1 vs training data — macro-F1 {res.f1_mean.mean():.3f}")
plt.tight_layout(); plt.savefig("/kaggle/working/per_species_f1.png", dpi=200)
res.to_csv("/kaggle/working/per_species_f1.csv", index=False)
print(res[["f1_mean", "train_n"]].corr(method="spearman"))
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = res.train_n.clip(lower=10)   # zero-training species drawn at x=10
ax.errorbar(x, res.f1_mean, yerr=res.f1_std, fmt="o", color="#3a7d44", ecolor="gray", alpha=0.8, ms=5)
z = res.train_n == 0
ax.scatter(x[z], res.f1_mean[z], marker="x", color="#c0392b", s=50, zorder=3, label="no training clips")
nudge = {"PHYNAT": (4, 8), "ADEDIP": (-40, -10), "LEPFUS": (4, -10), "SCIFUS": (6, 8), "SCINAS": (6, -10)}
for _, row in res.iterrows():
    if row.f1_mean < 0.6 or row.train_n < 400:
        ax.annotate(row.species, (max(row.train_n, 10), row.f1_mean), fontsize=7,
                    xytext=nudge.get(row.species, (4, 2)), textcoords="offset points")
ax.set_xscale("log"); ax.set_ylim(-0.05, 1.0); ax.legend(loc="lower right", fontsize=8)
ax.set_xlabel("Training clips (log scale)"); ax.set_ylabel("Test F1 (mean ± std, 3 seeds)")
ax.set_title(f"Per-species F1 vs training data (38 species, macro-F1 {res.f1_mean.mean():.3f}, Spearman ρ = 0.88)")
plt.tight_layout(); plt.savefig("/kaggle/working/per_species_f1.png", dpi=200); plt.show()